In [26]:
from qubo_qaoa.oriented_tangle.utils.qubo_utils import qubo_matrix_from_graph
from qubo_qaoa.oriented_tangle.utils.graph_utils import oriented_graph_with_copy_numbers
import numpy as np
from itertools import product


In [27]:
filepath = '/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/test_N2_W2.gfa'
copy_numbers = [1,1]

In [28]:
graph = oriented_graph_with_copy_numbers(filepath, copy_numbers)

In [29]:
nodes = list(graph.nodes)
V = int(len(nodes) / 2)
T = int(sum(graph.nodes[node]["weight"] for node in nodes) / 2)
print(f'V: {V}, T: {T}')

V: 2, T: 2


In [30]:
lambda_t = 10
lambda_g = 5
lambda_w = 1  

In [31]:
qubo_matrix = np.zeros((T, V, 2, T, V, 2), dtype=np.int16)
Q_walk_prime = np.ones((V, 2, V, 2), dtype=np.int8)
for i, sigma in product(range(V), range(2)):
    Q_walk_prime[i, sigma, i, sigma] = -1
Q_walk_prime *= lambda_t
for t in range(T):
    qubo_matrix[t, :, :, t, :, :] += Q_walk_prime
qt = qubo_matrix.reshape((T * V * 2), (T * V * 2))

In [47]:
v = [1,0,0,0,1,0,0,0]
v @ qt @ v + T * lambda_t

np.int64(0)

In [33]:
# Traverse graph edges constraints
qubo_matrix = np.zeros((T, V, 2, T, V, 2), dtype=np.int16)
Q_graph_prime = np.zeros((V, 2, V, 2), dtype=np.int8)

for i, sigma, j, tau in product(range(V), range(2), range(V), range(2)):
    if (nodes[2 * i + sigma], nodes[2 * j + tau]) in graph.edges:
        Q_graph_prime[i, sigma, j, tau] = -1 
Q_graph_prime *= lambda_g
g_offset = (T-1) * lambda_g

for t in range(T - 1):
    qubo_matrix[t,     :, :, t + 1, :, :] += Q_graph_prime
    
qg = qubo_matrix.reshape((T * V * 2), (T * V * 2))

In [34]:
qg

array([[ 0,  0,  0,  0,  0,  0, -5,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0, -5,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0]], dtype=int16)

In [48]:
v = [1,0,0,0,1,0,0,0]
v @ qg @ v + g_offset

np.int64(5)

In [39]:
# Number of visits constraints
qubo_matrix = np.zeros((T, V, 2, T, V, 2), dtype=np.int16)
for i in range(V):
    Q_weight_prime = np.ones((T, 2, T, 2), dtype=np.int8)
    for t, sigma in product(range(T), range(2)):
        Q_weight_prime[t, sigma, t, sigma] -= int(2 * graph.nodes[nodes[2 * i]]["weight"])

    qubo_matrix[:, i, :, :, i, :] += Q_weight_prime * lambda_w
qw = qubo_matrix.reshape((T * V * 2), (T * V * 2))
w_offset = lambda_w * int(sum(graph.nodes[nodes[2 * i]]["weight"] ** 2 for i in range(V)))

In [49]:
v = [1,0,0,0,1,0,0,0]
v @ qw @ v + w_offset

np.int64(2)